# Relation to Graph Mapping, Example 1
This example will provide an illustration of the relation to graph mapping and graph based analysis of a real dataset. This dataset is from the SBA. A mapping to the guidelines provided in the [relation to graph mapping](https://github.com/rajivsam/descriptive_analytics/blob/main/examples/graph_from_relations/rel-to-graph-concepts.pdf) is provided. 

## Initial Assessment
The following is done as part of initial assessment
1. Data quality assessment
2. Data analysis to remove data inconsistent with known business conditions
3. Data analysis to remove data with too little support to make valid inference
4. Remediation to fix data with little support by broadening codes that are hierarchical, like industry classification codes and zip codes
5. Defining new attributes as needed
6. Defining the attributes needed for analysis and dropping unnecessary attributes

### Use the utility classes to process the specification for relational to graph mapping
An example of using the relation to heterogeneous graph mapping is provided. The analysis component requires a homogeneous graph. Please see the analysis notebook for an example of relational to homogeneous graph mapping.

In [ ]:
import yaml
fp = "../config/sba_loans/sba_homog_explicit.yml"

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
from relation_to_graph.mapper.entity_filter import EntityFilter

In [ ]:
hm = EntityFilter(fp)

### A brief summary of the dataset
The small business administration (SBA) makes it possible for small businesses to obtain working capital. They gaurantee borowers, lenders use this guarantee to lend money to business owners. For more information, see [this page](https://www.sba.gov/funding-programs/loans/7a-loans). The SBA reports loan performance. Most loans are paid in full, a small fraction default. We can apply machine learning to determine which loans default. Please refer to the data dictionary in the data folder for detailed description of attributes in the dataset.

In [ ]:
import pandas as pd
dfr = hm.get_data_frame()

In [ ]:
dfr.head()

In [ ]:
len(dfr["BankFDICNumber"].unique())

In [ ]:
len(dfr["BorrName"].unique())

In [ ]:
dfr["LoanStatus"].unique()

In [ ]:
len(dfr["NaicsCode"].unique())

### Exclude small number of loans with unknown business implication
There are about 3 loans which have the first disbursement date that is later than the date the loan was pain in full. This is either a data issue or, more likely, a business condition that is codified this way

In [ ]:
dfr["FirstDisbursementDate"] = pd.to_datetime(dfr["FirstDisbursementDate"])
dfr["AsOfDate"] = pd.to_datetime(dfr["AsOfDate"])
dfr["PaidInFullDate"] = pd.to_datetime(dfr["PaidInFullDate"])
pna = (dfr["FirstDisbursementDate"].isna()) | (dfr["FirstDisbursementDate"] > dfr["PaidInFullDate"])

to_exclude = (dfr.LoanStatus == "EXEMPT") | (dfr.LoanStatus == "CANCLD") | (dfr.LoanStatus == "COMMIT") | pna
dfr = dfr[~ to_exclude]

### Compute Initial Imbalance

In [ ]:
num_PIF = dfr[dfr.LoanStatus == "PIF"].shape[0]
num_CHGOFF = dfr[dfr.LoanStatus == "CHGOFF"].shape[0]

In [ ]:
dfr["LoanStatus"].value_counts()

In [ ]:
pct_chgoff = (num_CHGOFF/dfr.shape[0])*100
pct_pif = (num_PIF/dfr.shape[0])* 100
print(f" percent charged off is {pct_chgoff:.2f} %, percent paid in full {pct_pif:.2f}%")

In [ ]:
from datetime import datetime

def diff_month(row):
    if row["LoanStatus"] == "PIF":
        return (row["PaidInFullDate"].year - row["FirstDisbursementDate"].year) * 12 + row["PaidInFullDate"].month - row["FirstDisbursementDate"].month
    return (row["AsOfDate"].year - row["FirstDisbursementDate"].year) * 12 + row["AsOfDate"].month - row["FirstDisbursementDate"].month

### Create an Identifier for the Loan
A loan is one of the entity abstractions. This does not have an identifier, so we create one.

In [ ]:
dfr["LoanID"] = dfr.apply(lambda row: "Loan-" + str(row.name), axis=1)

In [ ]:
dfr["LoanID"]

### Create an attribute for number of Payments
The number of payments made with the loan is a derived attribute.

In [ ]:
dfr["NumPmtsMade"] = dfr.apply(diff_month, axis=1)

In [ ]:
col_remove = ["FirstDisbursementDate", "AsOfDate", "PaidInFullDate"]
cols = [c for c in dfr.columns.tolist() if c not in col_remove]

In [ ]:
dfr = dfr[cols]

In [ ]:
dfr["NumPmtsMade"].describe()

In [ ]:
dfr

In [ ]:
dfr = dfr.reset_index(drop=True)

In [ ]:
dfr

In [ ]:
dfr.dtypes

In [ ]:
dfr.LoanStatus.value_counts()

### Fix Missing Values

In [ ]:
dfr.loc[dfr.SoldSecMrktInd.isna(), "SoldSecMrktInd"]= "UNKNOWN"

In [ ]:
def bad_FDICNumber(row):

    if pd.isna(row["BankFDICNumber"]):
        idstr = row["BankName"][:10] + "-" + str(row["BankZip"])
        row["BankFDICNumber"] = idstr
    if isinstance(row["BankFDICNumber"], float):
        row["BankFDICNumber"] = int(row["BankFDICNumber"])
    return row["BankFDICNumber"]
dfr["BankFDICNumber"] = dfr.apply(bad_FDICNumber, axis=1)

In [ ]:
columns_with_nan = dfr.columns[dfr.isna().any()]

print(columns_with_nan)

In [ ]:
null_FDIC = dfr.BankFDICNumber.isna()
dfr[null_FDIC]

### Correct Data Types

In [ ]:
dfr.loc[:, "BankFDICNumber"] = dfr["BankFDICNumber"].astype(str)
dfr.loc[:, "BankZip"] = dfr["BankZip"].astype(str)
dfr.loc[:, "BorrZip"] = dfr["BorrZip"].astype(str)
dfr.loc[:, "NaicsCode"] = dfr["NaicsCode"].astype(str)
dfr.loc[:, "SoldSecMrktInd"] = dfr["SoldSecMrktInd"].astype(str)
dfr.loc[:, "CollateralInd"] = dfr["CollateralInd"].astype(str)

In [ ]:
dfr.dtypes

###  Fix Support for Codes
Some of the Zip Codes and NAICS codes have insufficient support. We can fix this by considering only the first few characters of the code. These codes are hierarchical, so doing this merges hierarchically

In [ ]:
dfr["BorrZip"] = dfr.BorrZip.str[:2] + 3*"X" 

### Verify Support
Verify the zip codes and NAICS codes have value counts of at least 5 per category

In [ ]:
dfr["BorrZip"].value_counts()

### Fix Support for Cities
Many Borrower and Lender cities have only one or two entries. Collectively, an insufficient support category is created and these loans are bucketed there.

In [ ]:
dfr["BankCity"].value_counts()

In [ ]:
dfr["BorrCity"].value_counts()

In [ ]:
dfr["BankZip"] = dfr.BankZip.str[:3] + 2*"X" 
dfr["BankZip"].value_counts()

In [ ]:
insuff_supp_bank_city = [index for index, value in dfr["BankCity"].value_counts().items() if value < 5]
insuff_supp_borr_city = [index for index, value in dfr["BorrCity"].value_counts().items() if value < 5]

In [ ]:
recode_bank_city = dfr["BankCity"].isin(insuff_supp_bank_city)

recode_borr_city = dfr["BorrCity"].isin(insuff_supp_borr_city)



In [ ]:
dfr.loc[recode_borr_city, "BorrCity"] = "XXXX"
dfr.loc[recode_bank_city, "BankCity"] = "XXXX"

In [ ]:
dfr["BorrCity"].value_counts()

In [ ]:
dfr["BankCity"].value_counts()

In [ ]:
dfr["BankZip"].value_counts()

In [ ]:
drop_insuff_supp = dfr.BankZip == "51XXX"
dfr = dfr[~drop_insuff_supp]

In [ ]:
dfr.loc[:, "NaicsCode"] = dfr.NaicsCode.str[:2] + 4*"X" 

In [ ]:
dfr.NaicsCode.value_counts()

In [ ]:
insuff_supp_borr_zip = [index for index, value in dfr["BorrZip"].value_counts().items() if value < 5]

In [ ]:
insuff_supp_borr_zip

In [ ]:
dfr

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_stage1.csv"
dfr.to_csv(fp, index=False)

In [ ]:
import kmds
from kmds.ontology.kmds_ontology import *
from kmds.tagging.tag_types import ExploratoryTags

In [ ]:
loan_data_url = "https://www.sba.gov/partners/lenders/7a-loan-program/types-7a-loans"

In [ ]:
from kmds.ontology.intent_types import IntentType
exp_obs_list = []
observation_count :int = 1
e1 = ExploratoryObservation(namespace=onto)

In [ ]:
e1.finding = f"Data Sourcing: The data was obtained from {loan_data_url}. The small business administration (SBA) makes\
it possible for small businesses to obtain working capital. They gaurantee borowers, lenders use this guarantee\
to lend money to business owners. For more information, see this page. The SBA reports loan performance.\
Most loans are paid in full, a small fraction default. We can apply machine learning to determine which loans default.\
Please refer to the data dictionary in the data folder for detailed description of attributes in the dataset."
e1.finding_sequence = observation_count
e1.exploratory_observation_type = ExploratoryTags.DATA_QUALITY_OBSERVATION.value
e1.intent = IntentType.DATA_UNDERSTANDING.value
exp_obs_list.append(e1)

In [ ]:
e1.finding

In [ ]:
observation_count +=1
e2 = ExploratoryObservation(namespace=onto)
e2.finding = "Data Cleaning: Exclude small number of loans with unknown business implication\
There are about 3 loans which have the first disbursement date that is later than the date the loan was pain in full.\
This is either a data issue or, more likely, a business condition that is codified this way"
e2.finding_sequence = observation_count
e2.exploratory_observation_type = ExploratoryTags.DATA_QUALITY_OBSERVATION.value
e2.intent = IntentType.DATA_UNDERSTANDING.value
exp_obs_list.append(e2)

In [ ]:
observation_count += 1
e3 = ExploratoryObservation(namespace=onto)
e3.finding = "Data Cleaning: The Industry classification code, Zip Code, Borrower City and Bank Zip have \
category levels with less than 5 instances. For hierachical categories like ZipCode or NAICS code,\
maksing can be used to rollup these categories to the higher level. For flat structured categorical attributes\
these instances were dropped."
e3.finding_sequence = observation_count
e3.exploratory_observation_type = ExploratoryTags.DATA_QUALITY_OBSERVATION.value
e3.intent = IntentType.DATA_UNDERSTANDING.value
exp_obs_list.append(e3)

In [ ]:
observation_count += 1
e4 = ExploratoryObservation(namespace=onto)
prep_nb_fname = "sba_loans.ipynb"
interim_fname = "sba_loans_stage1.csv"
e4 = ExploratoryObservation(namespace=onto)
e4.finding = f"Data Cleaning: The notebook {prep_nb_fname} performs data cleaning and writes the intermediate file\
{interim_fname}. This intermediate file is used to prepare the datasets for machine learning. This is the data that is\
encoded as a homogeneous graph"
e4.finding_sequence = observation_count
e4.exploratory_observation_type = ExploratoryTags.DATA_QUALITY_OBSERVATION.value
e4.intent = IntentType.DATA_UNDERSTANDING.value
exp_obs_list.append(e4)
observation_count += 1
e5 = ExploratoryObservation(namespace=onto)
e5.finding = "Data Cleaning: The Industry classification code, Zip Code, Borrower City and Bank Zip have \
category levels with less than 5 instances. For hierachical categories like ZipCode or NAICS code,\
maksing can be used to rollup these categories to the higher level. For flat structured categorical attributes\
these instances were dropped."
e5.finding_sequence = observation_count
e5.exploratory_observation_type = ExploratoryTags.DATA_QUALITY_OBSERVATION.value
e5.intent = IntentType.DATA_UNDERSTANDING.value
exp_obs_list.append(e5)
observation_count += 1
e6 = ExploratoryObservation(namespace=onto)
e6.finding = "Sequence of analysis notebooks, raw data to classifier: (1)sba_loans.ipynb (2) sba_loan_numeric_enc.ipynb\
(3) sba_bad_borr_graph_ctor.ipynb (4) sba_bad_loans_classifier.ipynb. For interpretation, see references provided \
in the knowledge base for this example."
e6.finding_sequence = observation_count
e6.exploratory_observation_type = ExploratoryTags.DATA_QUALITY_OBSERVATION.value
e6.intent = IntentType.DATA_UNDERSTANDING.value
exp_obs_list.append(e6)


In [ ]:
kaw = KnowledgeExtractionExperimentationWorkflow("sba_7a_loans_modeling", namespace=onto)

In [ ]:
kaw.has_exploratory_observations = exp_obs_list

In [ ]:

from owlready2 import *
from kmds.utils.path_utils import get_package_kb_path
KNOWLEDGE_BASE = "../data/kmds/sba_loans_kb.xml"
onto.save(file=KNOWLEDGE_BASE, format="rdfxml")